# Novelty Triage Prototype

Pipeline: **OOD → Characterization → Uncertainty → Priority Ranking → Human Review → Revisit**.

The characterization stage uses an Expected-Gradient-style attribution approach inspired by [Axiomatic Attribution Priors and Expected Gradients](https://arxiv.org/abs/1906.10670).

The `animal` column in the incoming dataset is treated as hidden ground truth for evaluation only.

In [ ]:
from pathlib import Path
import importlib.util

script_path = Path('novelty_triage_prototype.py')
spec = importlib.util.spec_from_file_location('triage', script_path)
triage = importlib.util.module_from_spec(spec)
spec.loader.exec_module(triage)
print('Loaded prototype.')

In [ ]:
# Run the full pipeline
out = triage.run_pipeline(
    train_path=Path('animal_training_dataset(2).csv'),
    incoming_path=Path('animal_incoming_unknown_dataset(1).csv'),
    output_dir=Path('.')
)


In [ ]:
# Human review queue
queue_cols = [
    'observation_id', 'predicted_class', 'novelty_score',
    'uncertainty_score', 'priority_score', 'characterization'
]
out['queue'][queue_cols].head(20)

In [ ]:
# Pick an observation and inspect the complete decision trace.
obs_id = out['queue'].iloc[0]['observation_id']
triage.review_observation(obs_id, out['results'], out['history'])

In [ ]:
# Go back directly to the characterization stage.
triage.revisit_stage(obs_id, out['history'], stage='2_CHARACTERIZATION')

In [ ]:
# Or inspect the complete pruning history for the observation.
triage.revisit_stage(obs_id, out['history'])